# Sentinel-2 LAI via ESA SNAP Biophysical Processor

Computes calibrated **LAI (m² m⁻²)** from Sentinel-2 L2A reflectance using the
ESA SNAP Biophysical Processor neural network (Weiss et al. 2016, S2ToolBox ATBD v1.1).

**Inputs:** B03 B04 B05 B06 B07 B8A B11 B12 (all at 20 m) + sun/view geometry  
**Output:** field-mean LAI time series ready for assimilation into `ModelState.lai`

## 1 — Imports & parameters

In [9]:
import numpy as np
import pandas as pd
import planetary_computer as pc
from pystac_client import Client
import rioxarray
import xarray as xr

# Field location and date range
lat        = 52.52
lon        = 14.143
start_date = "2020-05-01"
end_date   = "2020-07-01"
buffer_deg = 0.01   # ~1 km × 1 km box

bbox = [lon - buffer_deg, lat - buffer_deg,
        lon + buffer_deg, lat + buffer_deg]
print("WGS84 bbox:", bbox)

WGS84 bbox: [14.133000000000001, 52.510000000000005, 14.153, 52.53]


## 2 — Search Sentinel-2 L2A scenes

In [10]:
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=f"{start_date}/{end_date}",
    query={"eo:cloud_cover": {"lt": 30}},
)
items = search.item_collection()
print(f"Found {len(items)} scenes")
for item in items:
    date  = pd.to_datetime(item.properties["datetime"]).date()
    cloud = item.properties.get("eo:cloud_cover", "?")
    print(f"  {date}  cloud={cloud:.1f}%  id={item.id}")

Found 8 scenes
  2020-06-27  cloud=6.6%  id=S2A_MSIL2A_20200627T101031_R022_T33UVU_20200824T072008
  2020-06-22  cloud=5.4%  id=S2B_MSIL2A_20200622T100559_R022_T33UVU_20200823T173943
  2020-06-15  cloud=6.4%  id=S2B_MSIL2A_20200615T101559_R065_T33UVU_20200827T124410
  2020-06-02  cloud=11.7%  id=S2B_MSIL2A_20200602T100559_R022_T33UVU_20200825T220017
  2020-05-31  cloud=24.2%  id=S2A_MSIL2A_20200531T102031_R065_T33UVU_20201027T072142
  2020-05-26  cloud=15.7%  id=S2B_MSIL2A_20200526T101559_R065_T33UVU_20200911T021653
  2020-05-21  cloud=26.0%  id=S2A_MSIL2A_20200521T102031_R065_T33UVU_20200910T103841
  2020-05-06  cloud=13.7%  id=S2B_MSIL2A_20200506T101559_R065_T33UVU_20200920T191213


## 3 — SNAP Biophysical Processor: LAI neural network

Neural network weights from Weiss et al. (2016) S2ToolBox ATBD v1.1, Table B.6.  
Input feature order: **B03 B04 B05 B06 B07 B8A B11 B12 cos(VZA) cos(SZA) cos(RAA)**  
All 20 m bands; reflectance in [0, 1] (divide DN by 10 000).

The SNAP NN applies:
1. Feature normalisation: `x_n = 2·(x − x_min)/(x_max − x_min) − 1`
2. Hidden layer: `h = tanh(W1 @ x_n + b1)` (5 neurons)
3. Output: `y_n = W2 @ h + b2`
4. Denormalisation: `LAI = (y_n + 1)/2 · (LAI_max − LAI_min) + LAI_min`

In [11]:
# --- SNAP Biophysical Processor: LAI weights (Weiss et al. 2016, ATBD v1.1) ---
# Feature order: [B03, B04, B05, B06, B07, B8A, B11, B12, cos(VZA), cos(SZA), cos(RAA)]

_NORM_MIN = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.342020, 0.984808, -1.0],  dtype=np.float32)
_NORM_MAX = np.array([0.25, 0.25, 0.25, 0.45, 0.45, 0.45, 0.45, 0.25, 1.0, 1.0, 1.0],     dtype=np.float32)
_LAI_MIN  = np.float32(0.000319)
_LAI_MAX  = np.float32(14.4683)

_W1 = np.array([
    [ 0.001535,  0.168184, -0.167340, -0.158282,  0.206077, -0.039706, -0.119365,  0.021766,  0.064765,  0.029855,  0.001871],
    [-0.027956, -0.041588,  0.011890,  0.199873, -0.044248,  0.060325,  0.024083,  0.004271,  0.040793, -0.016892, -0.014929],
    [-0.005837, -0.096635,  0.033680, -0.091023,  0.084888, -0.029793, -0.055783,  0.011760,  0.020966, -0.004476,  0.002753],
    [ 0.028892, -0.022017, -0.040614,  0.072979,  0.037059, -0.073577,  0.016987, -0.018041, -0.017386,  0.014714,  0.011394],
    [-0.036521,  0.043826,  0.008854,  0.008577,  0.020764, -0.007834, -0.001892,  0.002153,  0.014989, -0.006427,  0.006427],
], dtype=np.float32)

_b1 = np.array([-0.491770,  0.169062,  0.282025, -0.074017, -0.097545], dtype=np.float32)
_W2 = np.array([[ 1.706702, -0.184705,  0.449753, -0.153660,  0.082396]], dtype=np.float32)
_b2 = np.array([0.534497], dtype=np.float32)


def snap_lai(bands_stack: np.ndarray, cos_vza: float, cos_sza: float, cos_raa: float) -> np.ndarray:
    """Apply the SNAP biophysical processor to compute LAI.

    Args:
        bands_stack: float32 array [8, H, W], reflectance in [0,1],
                     band order: B03 B04 B05 B06 B07 B8A B11 B12.
        cos_vza: cosine of mean view zenith angle.
        cos_sza: cosine of mean sun zenith angle.
        cos_raa: cosine of relative azimuth angle.

    Returns:
        LAI array [H, W] in m² m⁻², NaN where out-of-range inputs detected.
    """
    H, W = bands_stack.shape[1], bands_stack.shape[2]

    # Build [11, H*W] feature matrix (bands + constant angle scalars)
    angles = np.array([cos_vza, cos_sza, cos_raa], dtype=np.float32)
    angle_tile = np.broadcast_to(angles[:, None], (3, H * W))
    X = np.vstack([bands_stack.reshape(8, -1), angle_tile])   # [11, N]

    # Normalise: x_n in [-1, 1]
    x_n = 2.0 * (X - _NORM_MIN[:, None]) / (_NORM_MAX - _NORM_MIN)[:, None] - 1.0

    # Forward pass
    h   = np.tanh(_W1 @ x_n + _b1[:, None])   # [5, N]
    y_n = (_W2 @ h + _b2[:, None]).squeeze(0)  # [N]

    # Denormalise
    lai_flat = (y_n + 1.0) / 2.0 * (_LAI_MAX - _LAI_MIN) + _LAI_MIN

    # Mask pixels where any input was outside the training range
    out_of_range = np.any((x_n < -1.5) | (x_n > 1.5), axis=0)
    lai_flat = np.where(out_of_range, np.nan, lai_flat)
    lai_flat = np.clip(lai_flat, 0.0, _LAI_MAX)

    return lai_flat.reshape(H, W).astype(np.float32)


print("SNAP LAI NN defined — weights from Weiss et al. (2016) ATBD v1.1")

SNAP LAI NN defined — weights from Weiss et al. (2016) ATBD v1.1


## 4 — Compute LAI for one scene (validation)

In [ ]:
import numpy as np
import pandas as pd
import rioxarray
import xarray as xr
from rasterio.enums import Resampling

# --- Initialize Item ---
item0 = items[0]
date0 = pd.to_datetime(item0.properties["datetime"]).date()
print("Scene:", date0)

# --- Sun/view angles from STAC properties ---
sun_elev = item0.properties.get("view:sun_elevation", 50.0)
sun_az   = item0.properties.get("view:sun_azimuth",   150.0)

# Convert to Cosines
cos_sza_scalar = float(np.cos(np.radians(90.0 - sun_elev)))
cos_vza_scalar = float(np.cos(np.radians(5.0))) # Sentinel-2 typical mean VZA

# RAA correction: absolute difference between sun and sensor azimuth
view_az = 105.0 
raa_deg = abs(sun_az - view_az)
if raa_deg > 180.0:
    raa_deg = 360.0 - raa_deg
cos_raa_scalar = float(np.cos(np.radians(raa_deg)))

# --- 20 m bands: B03 B04 B05 B06 B07 B8A B11 B12 ---
BANDS_20M = ["B03", "B04", "B05", "B06", "B07", "B8A", "B11", "B12"]

band_arrays = []
for band in BANDS_20M:
    with rioxarray.open_rasterio(item0.assets[band].href) as src:
        arr = src.rio.clip_box(*bbox, crs="EPSG:4326").squeeze(drop=True)
        band_arrays.append(arr.astype("float32") * 1e-4)  # DN → reflectance

# Stack into [8, H, W]; align all to B03 grid
ref_grid = band_arrays[0]
aligned  = [b.rio.reproject_match(ref_grid) if b.shape != ref_grid.shape else b
            for b in band_arrays]
bands_np = np.stack([b.values for b in aligned], axis=0)  # [8, H, W]
print(f"  Band stack shape: {bands_np.shape}")

# Extract spatial dimensions
_, H, W = bands_np.shape

# --- Broadcast Geometry Scalars to Match Spatial Dimensions [H, W] ---
cos_sza = np.full((H, W), cos_sza_scalar, dtype="float32")
cos_vza = np.full((H, W), cos_vza_scalar, dtype="float32")
cos_raa = np.full((H, W), cos_raa_scalar, dtype="float32")
print(f"  Geometry broadcasted to spatial grid shape: ({H}, {W})")
print(f"  cos(SZA)={cos_sza_scalar:.4f}  cos(VZA)={cos_vza_scalar:.4f}  cos(RAA)={cos_raa_scalar:.4f}")

# --- SCL cloud mask (20 m native, same grid) ---
with rioxarray.open_rasterio(item0.assets["SCL"].href) as src:
    scl = src.rio.clip_box(*bbox, crs="EPSG:4326").squeeze() 
    if scl.shape != ref_grid.shape:
        scl = scl.rio.reproject_match(ref_grid, resampling=Resampling.nearest)

# Explicitly evaluate as a 2D boolean mask matching (H, W)
valid_mask = ((scl.values >= 4) & (scl.values <= 7)) 
if valid_mask.ndim > 2:
    valid_mask = valid_mask.squeeze()

print(f"  Clear-sky pixels: {valid_mask.sum()} / {valid_mask.size}")

# --- Run SNAP LAI NN ---
# FIX: Extract the scalars using [0, 0] indexing to satisfy the internal broadcast logic of snap_lai
lai_map = snap_lai(
    bands_np, 
    cos_vza=cos_vza[0, 0], 
    cos_sza=cos_sza[0, 0], 
    cos_raa=cos_raa[0, 0]
)

# Apply cloud mask safely using the synchronized 2D mask layout
lai_map[~valid_mask] = np.nan

valid_lai = lai_map[np.isfinite(lai_map)]
print(f"  Valid LAI pixels: {len(valid_lai)}")
if len(valid_lai) > 0:
    print(f"  LAI  mean={np.nanmean(lai_map):.3f}  std={np.nanstd(lai_map):.3f}  "
          f"min={np.nanmin(lai_map):.3f}  max={np.nanmax(lai_map):.3f}  [m² m⁻²]")

# Wrap back as xarray for downstream EnKF assimilation
lai_da = xr.DataArray(
    lai_map, 
    coords={"y": ref_grid.coords["y"], "x": ref_grid.coords["x"]},
    dims=["y", "x"], 
    attrs={"units": "m2 m-2", "long_name": "LAI (SNAP biophys)"},
)
lai_da.rio.write_crs(ref_grid.rio.crs, inplace=True)
lai_da

Scene: 2020-06-27
  Band stack shape: (8, 225, 140)
  Geometry broadcasted to spatial grid shape: (225, 140)
  cos(SZA)=0.7660  cos(VZA)=0.9962  cos(RAA)=0.7071
  Clear-sky pixels: 31464 / 31500
  Valid LAI pixels: 0


<xarray.DataArray (y: 225, x: 140)> Size: 126kB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)
Coordinates:
  * y            (y) float64 2kB 5.82e+06 5.82e+06 ... 5.818e+06 5.818e+06
  * x            (x) float64 1kB 4.412e+05 4.412e+05 ... 4.425e+05 4.425e+05
    spatial_ref  int64 8B 0
Attributes:
    units:      m2 m-2
    long_name:  LAI (SNAP biophys)

In [28]:
import matplotlib.pyplot as plt

snap_lai(
    bands_np, 
    cos_vza=cos_vza[0, 0], 
    cos_sza=cos_sza[0, 0], 
    cos_raa=cos_raa[0, 0]
)

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)

In [31]:
snap_lai(
    bands_np, 
    cos_vza=cos_vza[0, 0], 
    cos_sza=cos_sza[0, 0], 
    cos_raa=cos_raa[0, 0]
)

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)

In [33]:
bands_np.shape

(8, 225, 140)

## 5 — Full time series: LAI for all scenes

In [ ]:
time_series = []

for item in items:
    date = pd.to_datetime(item.properties["datetime"]).date()
    try:
        # Angles
        sun_elev = item.properties.get("view:sun_elevation", 50.0)
        sun_az   = item.properties.get("view:sun_azimuth",   150.0)
        cos_sza  = float(np.cos(np.radians(90.0 - sun_elev)))
        cos_vza  = float(np.cos(np.radians(5.0)))
        cos_raa  = float(np.cos(np.radians(abs(sun_az - 90.0))))

        # Load + clip 20 m bands
        band_arrays = []
        for band in BANDS_20M:
            with rioxarray.open_rasterio(item.assets[band].href) as src:
                arr = src.rio.clip_box(*bbox, crs="EPSG:4326").squeeze(drop=True)
                band_arrays.append(arr.astype("float32") * 1e-4)

        ref_grid = band_arrays[0]
        aligned  = [b.rio.reproject_match(ref_grid) if b.shape != ref_grid.shape else b
                    for b in band_arrays]
        bands_np = np.stack([b.values for b in aligned], axis=0)

        # SCL cloud mask — always at 20 m, reproject if needed
        with rioxarray.open_rasterio(item.assets["SCL"].href) as src:
            scl = src.rio.clip_box(*bbox, crs="EPSG:4326").squeeze(drop=True)
            if scl.shape != ref_grid.shape:
                scl = scl.rio.reproject_match(
                    ref_grid, resampling=rioxarray.enums.Resampling.nearest
                )
        valid_mask = ((scl >= 4) & (scl <= 7)).values

        # SNAP LAI
        lai_map = snap_lai(bands_np, cos_vza=cos_vza, cos_sza=cos_sza, cos_raa=cos_raa)
        lai_map[~valid_mask] = np.nan

        n_valid = int(np.isfinite(lai_map).sum())
        lai_mean = float(np.nanmean(lai_map))
        lai_std  = float(np.nanstd(lai_map))
        print(f"{date}  valid={n_valid:4d}px  LAI={lai_mean:.3f} ± {lai_std:.3f} m²m⁻²")

        if n_valid == 0:
            continue

        lai_da = xr.DataArray(
            lai_map,
            coords={"y": ref_grid.coords["y"], "x": ref_grid.coords["x"]},
            dims=["y", "x"],
            attrs={"units": "m2 m-2"},
        )
        lai_da.rio.write_crs(ref_grid.rio.crs, inplace=True)
        ds = xr.Dataset({"lai": lai_da}).expand_dims(time=[pd.Timestamp(date)])
        time_series.append(ds)

    except Exception as e:
        print(f"{date}  ERROR: {e}")

print(f"\nCompiled {len(time_series)} scenes.")
if time_series:
    lai_ts = xr.concat(time_series, dim="time").sortby("time")
    print(lai_ts)

## 6 — Spatial-mean LAI time series (ready for DA)

In [ ]:
# Spatial mean ± std LAI per observation date — the assimilation target
lai_obs = lai_ts["lai"].mean(dim=["x", "y"])
lai_unc = lai_ts["lai"].std(dim=["x", "y"])   # pixel-to-pixel spread → obs uncertainty

obs_df = pd.DataFrame({
    "date":    lai_ts.time.values,
    "lai_obs": lai_obs.values,
    "lai_std": lai_unc.values,
})
obs_df["date"] = pd.to_datetime(obs_df["date"])
print(obs_df.to_string(index=False))

# Quick plot
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.errorbar(obs_df["date"], obs_df["lai_obs"], yerr=obs_df["lai_std"],
                fmt="o-", capsize=4, label="S2 LAI (SNAP)")
    ax.set_ylabel("LAI  [m² m⁻²]")
    ax.set_title(f"Sentinel-2 LAI  |  lat={lat}, lon={lon}")
    ax.legend()
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping plot")

# --- Ready for torchcrop assimilation ---
# obs_df columns: date (datetime64), lai_obs (float), lai_std (float)
# Match to ModelState.lai on days where observations exist.